In [0]:
%python
from pyspark.sql import Row

data = [
    Row(name="Alice", age=29 , email="alice@example.com"),
    Row(name="Bob", age=35 , email="bob@example.com"),
    Row(name="Carol", age=42, email="carol@example.com")
]

df = spark.createDataFrame(data)
# df.filter(df.age > 30).display()
df.display()

In [0]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType
import re

# Define a Python function to mask email addresses
# e.g. "alice@example.com" -> "a*****e@example.com"
def mask_email(email):
    if email is None or email == "":
        return email
    if not re.match(r'^[^@]+@[^@]+\.[^@]+$', email):
        return '***'
    local, domain = email.split('@', 1)
    if len(local) <= 2:
        return local + '@' + domain
    return local[0] + '*****' + local[-1] + '@' + domain

# Register as a Spark UDF
mask_email_udf = udf(mask_email, StringType())

# Apply the mask to the email column
masked_df = df.withColumn("masked_email", mask_email_udf(col("email")))
masked_df.display()